# DS 4002 – Project 3: Classifying Leukemic B-Lymphoblast Cells
**Group:** Model Citizens &nbsp;|&nbsp; **Members:** Neil Parikh, Shaina Banduri, Nishana Dahal

---
## Step 1: Preprocessing

In [1]:
import os
import pathlib
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, UnidentifiedImageError

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

### 1.1 Dataset paths

In [2]:
DATA_ROOT = pathlib.Path(
    "/Users/NeilParikh/Desktop/ds4002/Project 3/C-NMC 2019 (PKG)"
)

TRAIN_ROOT = DATA_ROOT / "C-NMC_training_data"
TEST_PRELIM = DATA_ROOT / "C-NMC_test_prelim_phase_data"
TEST_FINAL  = DATA_ROOT / "C-NMC_test_final_phase_data"

FOLDS = ["fold_0", "fold_1", "fold_2"]
CLASSES = {"all": 1, "hem": 0}   # all = leukemia (positive), hem = normal (negative)

# ImageNet normalisation constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

print("Data root exists:", DATA_ROOT.exists())

Data root exists: True


### 1.2 Audit: scan for corrupted or unloadable images
### TAKES 35 MINTUES TO RUN LOCALLY -NEIL ###

In [ ]:
def collect_labeled_samples(train_root, folds, classes):
    """Return a list of (path, label) tuples from all training folds."""
    samples = []
    for fold in folds:
        for cls_name, label in classes.items():
            cls_dir = train_root / fold / cls_name
            for img_path in sorted(cls_dir.glob("*.bmp")):
                samples.append((img_path, label))
    return samples


def audit_images(samples):
    """Try to open every image; return (good, bad) lists."""
    good, bad = [], []
    for path, label in samples:
        try:
            with Image.open(path) as img:
                img.verify()           # checks file integrity without full decode
            good.append((path, label))
        except (UnidentifiedImageError, OSError, SyntaxError) as e:
            bad.append((path, label, str(e)))
    return good, bad


all_samples = collect_labeled_samples(TRAIN_ROOT, FOLDS, CLASSES)
print(f"Total labeled samples found : {len(all_samples):,}")

RUN_AUDIT = False

if RUN_AUDIT:
    good_samples, bad_samples = audit_images(all_samples)
    print(f"  Loadable images            : {len(good_samples):,}")
    print(f"  Corrupted / unloadable     : {len(bad_samples)}")
    if bad_samples:
        print("\nCorrupted files:")
        for p, lbl, err in bad_samples:
            print(f"  {p.name}  [{err}]")
    else:
        print("\nNo corrupted files detected — dataset is clean.")
else:
    good_samples = all_samples
    print(f"Audit skipped — using all {len(good_samples):,} samples.")

Total labeled samples found : 10,661
  Loadable images            : 10,661
  Corrupted / unloadable     : 0

No corrupted files detected — dataset is clean.


### 1.3 Class distribution & inverse-frequency weights

In [3]:
label_counts = Counter(label for _, label in good_samples)
n_hem = label_counts[0]   # normal
n_all = label_counts[1]   # leukemia
n_total = len(good_samples)

print("Class distribution (training set)")
print(f"  hem  (normal / label 0) : {n_hem:,}  ({100*n_hem/n_total:.1f}%)")
print(f"  all  (leukemia / label 1): {n_all:,}  ({100*n_all/n_total:.1f}%)")
print(f"  Total                    : {n_total:,}")
print(f"  Imbalance ratio (all:hem): {n_all/n_hem:.2f}:1")

# Inverse-frequency weights for weighted cross-entropy loss
# weight_c = N / (num_classes * count_c)
class_weights = torch.tensor(
    [n_total / (2 * n_hem), n_total / (2 * n_all)],
    dtype=torch.float32
)
print(f"\nInverse-frequency class weights (for CrossEntropyLoss)")
print(f"  hem  (index 0): {class_weights[0]:.4f}")
print(f"  all  (index 1): {class_weights[1]:.4f}")

NameError: name 'good_samples' is not defined

### 1.4 Define transforms (training vs. validation/test)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Training transform pipeline:")
print(train_transform)
print("\nVal / Test transform pipeline:")
print(val_test_transform)

Training transform pipeline:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Val / Test transform pipeline:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


### 1.5 Dataset class

In [ ]:
#this pulls from the separate dataset.py script

from dataset import CnmcDataset
print("CnmcDataset imported.")

CnmcDataset class defined.


### 1.6 Build fold-based train / validation split
We use **fold_0 + fold_1** for training and **fold_2** for validation (this will rotate during model selection).

In [8]:
def build_split(train_root, train_folds, val_folds, classes):
    """Return (train_samples, val_samples) given fold assignments."""
    train_samples = collect_labeled_samples(train_root, train_folds, classes)
    val_samples   = collect_labeled_samples(train_root, val_folds,   classes)
    return train_samples, val_samples


train_samples, val_samples = build_split(
    TRAIN_ROOT,
    train_folds=["fold_0", "fold_1"],
    val_folds=["fold_2"],
    classes=CLASSES,
)

print(f"Training samples   : {len(train_samples):,}")
print(f"Validation samples : {len(val_samples):,}")

train_counts = Counter(lbl for _, lbl in train_samples)
val_counts   = Counter(lbl for _, lbl in val_samples)
print(f"\nTrain  — hem: {train_counts[0]:,}  |  all: {train_counts[1]:,}")
print(f"Val    — hem: {val_counts[0]:,}   |  all: {val_counts[1]:,}")

Training samples   : 7,108
Validation samples : 3,553

Train  — hem: 2,293  |  all: 4,815
Val    — hem: 1,096   |  all: 2,457


In [9]:
BATCH_SIZE = 32

train_dataset = CnmcDataset(train_samples, transform=train_transform)
val_dataset   = CnmcDataset(val_samples,   transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader):,} batches of {BATCH_SIZE}")
print(f"Val   loader: {len(val_loader):,} batches of {BATCH_SIZE}")

Train loader: 223 batches of 32
Val   loader: 112 batches of 32


### 1.7 Sanity check: inspect one batch

In [10]:
images_batch, labels_batch = next(iter(train_loader))

print("Batch tensor shape :", images_batch.shape)   # expected: (32, 3, 224, 224)
print("Labels shape       :", labels_batch.shape)   # expected: (32,)
print("Pixel value range  : [{:.3f}, {:.3f}]".format(images_batch.min().item(), images_batch.max().item()))
print("Dtype              :", images_batch.dtype)
print("Labels in batch    :", labels_batch.unique().tolist())

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=77, pipe_handle=91)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'CnmcDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


KeyboardInterrupt: 

### 1.8 Visual confirmation: display sample preprocessed images

In [ ]:
LABEL_NAMES = {0: "hem (normal)", 1: "all (leukemia)"}

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD ).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


n_show = 8
fig, axes = plt.subplots(2, n_show // 2, figsize=(14, 6))
axes = axes.flatten()

for i in range(n_show):
    img = denormalize(images_batch[i]).permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(LABEL_NAMES[labels_batch[i].item()], fontsize=9)
    axes[i].axis("off")

fig.suptitle(
    f"Sample preprocessed images (resized to {IMG_SIZE}×{IMG_SIZE}, ImageNet-normalised)",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()
print("Preprocessing complete — images are correctly resized, normalised, and loading via DataLoader.")